Send Fraud Alert Email
Checks for newly flagged customer accounts (shipment/return round-trip pattern above threshold) and emails an alert only when something new appears.

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_json_df
import pandas as pd
import json
import io
import datetime
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery/analysis"
PREVIOUS_FLAGS_PATH = f"{ANALYSIS_BASE}/fraud_flagged_customers_previous.json"
print(f"Tracking file path: {PREVIOUS_FLAGS_PATH}")

Load current flagged candidates and previous run's list

In [0]:
customer_fraud_summary = read_json_df(blob_service, f"{ANALYSIS_BASE}/fraud_customer_summary.json")

flagged_candidates = customer_fraud_summary[
    (customer_fraud_summary["total_month_end_shipments"] >= fraud_min_shipments) &
    (customer_fraud_summary["round_trip_rate"] >= fraud_rate_threshold)
].sort_values("round_trip_rate", ascending=False)

current_flagged_ids = set(flagged_candidates["customer_no_shipment"].astype(str))
print(f"Currently flagged: {len(current_flagged_ids)}")

try:
    blob_client = blob_service.get_blob_client(container="gold", blob=PREVIOUS_FLAGS_PATH)
    raw = blob_client.download_blob().readall()
    previous_flagged = set(str(x) for x in json.loads(raw))
    print(f"Successfully loaded previous flags: {len(previous_flagged)} accounts")
except Exception as e:
    print(f"COULD NOT LOAD previous flags - treating as first run. Error: {type(e).__name__}: {e}")
    previous_flagged = set()

new_flags = current_flagged_ids - previous_flagged
print(f"New flagged accounts since last run: {len(new_flags)}")
print(new_flags)

In [0]:
print(f"RUN TIMESTAMP: {datetime.datetime.now()}")
print(f"Reading from: gold/{PREVIOUS_FLAGS_PATH}")
print(f"Current flagged IDs ({len(current_flagged_ids)}): {sorted(current_flagged_ids)}")
print(f"Previous flagged IDs ({len(previous_flagged)}): {sorted(previous_flagged)}")
print(f"New: {sorted(new_flags)}")

In [0]:
print("Current flagged IDs:", current_flagged_ids)
print("Previous flagged IDs:", previous_flagged)
print("Overlap:", current_flagged_ids & previous_flagged)

namal_check = flagged_candidates[flagged_candidates["customer_name"] == "NAMAL TYRE HOUSE"]
print("\nNAMAL TYRE HOUSE ID today:")
print(namal_check[["customer_no_shipment", "customer_name"]])

Only send if there's something new

In [0]:
if new_flags:
    new_flag_rows = flagged_candidates[flagged_candidates["customer_no_shipment"].astype(str).isin(new_flags)]

    salesperson_rollup = new_flag_rows.groupby("primary_salesperson").agg(
        flagged_accounts=("customer_name", "count"),
        total_value=("total_value_involved", "sum")
    ).sort_values("flagged_accounts", ascending=False)

    body = f"""Hello,

The automated fraud-pattern check has identified {len(new_flags)} new customer account(s) showing an elevated shipment/return round-trip rate near month-end (potential bonus-gaming pattern), above the {fraud_rate_threshold:.1%} threshold.

BY SALESPERSON
"""
    for sp, row in salesperson_rollup.iterrows():
        body += f"{sp}: {int(row['flagged_accounts'])} flagged accounts, {row['total_value']:,.2f} total value\n"

    body += "\nNEWLY FLAGGED ACCOUNTS\n"
    for _, row in new_flag_rows.iterrows():
        body += (
            f"- {row['customer_name']} (salesperson: {row['primary_salesperson']}, "
            f"{row['distinct_salespeople']} distinct): "
            f"{row['round_trip_count']} round-trips of {row['total_month_end_shipments']} month-end shipments "
            f"({row['round_trip_rate']:.1%}), total value {row['total_value_involved']:,.2f}\n"
        )

    body += """
Please review the attached evidence file for transaction-level detail before taking any action. This is a pattern flag for investigation, not a confirmed finding.

This is an automated message from the Exide Sales Fraud Detection pipeline.
"""

    msg = MIMEMultipart()
    msg["From"] = smtp_username
    msg["To"] = ", ".join(fraud_alert)
    msg["Subject"] = f"⚠ Fraud Pattern Alert - {len(new_flags)} New Flagged Account(s) - {datetime.date.today().isoformat()}"
    msg.attach(MIMEText(body, "plain"))

    blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/fraud_flagged_evidence.xlsx")
    evidence_bytes = blob_client.download_blob().readall()
    attachment = MIMEApplication(evidence_bytes, _subtype="xlsx")
    attachment.add_header("Content-Disposition", "attachment", filename="fraud_flagged_evidence.xlsx")
    msg.attach(attachment)

    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(smtp_username, smtp_password)
        server.sendmail(smtp_username, fraud_alert, msg.as_string())
        print(f"Alert email sent to {', '.join(fraud_alert)}")
    except Exception as e:
        print(f"Failed to send alert email: {type(e).__name__}: {e}")
    finally:
        server.quit()
else:
    print("No new flagged accounts - no email sent")

Update the "previous run" record regardless

In [0]:
all_flagged_ids = list(current_flagged_ids)

try:
    blob_client = blob_service.get_blob_client(container="gold", blob=PREVIOUS_FLAGS_PATH)
    blob_client.upload_blob(json.dumps(all_flagged_ids), overwrite=True)
    print(f"Successfully saved tracking list: {len(all_flagged_ids)} currently flagged accounts to {PREVIOUS_FLAGS_PATH}")
except Exception as e:
    print(f"FAILED to save tracking list: {type(e).__name__}: {e}")

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=PREVIOUS_FLAGS_PATH)
verify_content = json.loads(blob_client.download_blob().readall())
print(f"Verification read-back: {len(verify_content)} accounts saved")
print(f"Matches what we just saved: {set(str(x) for x in verify_content) == current_flagged_ids}")